# Stage 2: preference training (DPO) on the ranked pairs

Stage 1 taught the model to copy the teacher's briefings. This stage teaches it a comparison: for the same packet, *this* briefing is better than *that* one. The pairs come from `training/rank_pairs.py` (hard rules first, then a judge asked twice with the order swapped). The loss, **DPO**, pushes the model toward the chosen briefing and away from the rejected one, relative to a frozen copy of the stage 1 model so it cannot drift far.

Inputs: `MyDrive/news-briefer/news-briefer-lora` (the stage 1 adapter) and the pair set named by `PAIRS_NAME` (fetched from the GitHub release if it is not in Drive). Output: the adapter named by `OUTPUT_NAME`, plus its training log, which the last cell downloads.

**Runtime → Change runtime type → L4 GPU**, then **Runtime → Run all**. About 20 minutes on an L4 for a few hundred pairs.


In [ ]:
import subprocess, sys, importlib

USE_UNSLOTH = True  # the fallback cell sets this to False

def pip(*args):
    cmd = [sys.executable, "-m", "pip", "install", "--quiet", *args]
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-3000:], result.stderr[-3000:])
        raise RuntimeError("pip failed -- run the fallback cell below instead")

# unsloth pins its own tested versions of trl / peft / transformers / accelerate /
# bitsandbytes / datasets, so we pin unsloth to a release series and let it choose.
pip("--upgrade", "unsloth==2025.9.*", "unsloth_zoo==2025.9.*")
pip("matplotlib")

print()
print("installed versions:")
for name in ["unsloth", "trl", "peft", "transformers", "datasets", "accelerate", "bitsandbytes", "torch"]:
    try:
        mod = importlib.import_module(name)
        print(f"  {name:<14} {getattr(mod, '__version__', '?')}")
    except Exception as exc:  # noqa: BLE001
        print(f"  {name:<14} NOT IMPORTABLE ({exc})")

## 2. The pairs and the stage 1 adapter

In [ ]:
import json, os, collections
from google.colab import drive, files

drive.mount("/content/drive")
WORK_DIR = "/content/drive/MyDrive/news-briefer"
PAIRS_NAME = "dpo_pairs_r1.jsonl"          # the pair set to train on (stage 3 round 1: training/build_pairs_v3.py)
START_FROM = "news-briefer-dpo2-lora"     # the adapter this pass starts from
OUTPUT_NAME = "news-briefer-r1-lora"      # where the trained adapter goes
LEARNING_RATE = 2e-5                      # 4x the stage 2 passes, which barely moved the model
ADAPTER_DIR = os.path.join(WORK_DIR, START_FROM)
PAIRS = os.path.join(WORK_DIR, PAIRS_NAME)
assert os.path.isdir(ADAPTER_DIR), f"starting adapter not found at {ADAPTER_DIR}"
if not os.path.exists(PAIRS):
    # The ranked pairs are published as a release asset, so no upload is needed.
    import urllib.request
    urllib.request.urlretrieve(
        "https://github.com/sanjivp2703/adaptive-news-briefing-agent/releases/download/training-data-v7/" + PAIRS_NAME,
        PAIRS,
    )
    print("downloaded", PAIRS_NAME, "into Drive")

with open(PAIRS, encoding="utf-8") as fh:
    pairs = [json.loads(l) for l in fh if l.strip()]
print(len(pairs), "pairs;", dict(collections.Counter(p.get("source") or p.get("decided_by") for p in pairs)))
print("by profile:", dict(collections.Counter(p.get("profile") for p in pairs)))


## 3. Load the stage 1 model and prepare the pairs

Each pair becomes a prompt (the same system + user messages as training), a chosen answer and a rejected answer, rendered through the chat template. The reference model is the stage 1 model with the adapter switched off during the comparison, which is how TRL handles a LoRA policy without a second copy in memory.

In [ ]:
import torch
from unsloth import FastLanguageModel, PatchDPOTrainer
PatchDPOTrainer()

model, tokenizer = FastLanguageModel.from_pretrained(model_name=ADAPTER_DIR, max_seq_length=8192, dtype=None, load_in_4bit=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

from datasets import Dataset
def render(p):
    prompt = tokenizer.apply_chat_template(p["prompt"], tokenize=False, add_generation_prompt=True)
    end = tokenizer.eos_token or ""
    return {"prompt": prompt, "chosen": p["chosen"].strip() + end, "rejected": p["rejected"].strip() + end}
ds = Dataset.from_list([render(p) for p in pairs]).shuffle(seed=3407)
split = ds.train_test_split(test_size=0.1, seed=3407)
train_ds, eval_ds = split["train"], split["test"]
print(len(train_ds), "train pairs,", len(eval_ds), "held-out pairs")


## 4. Train

`beta` sets how far the model may move from the stage 1 reference: 0.1 is the usual starting point. The learning rate is set at the top of the notebook; the stage 2 passes used 5e-6 and barely moved the model, so stage 3 rounds start at 2e-5. Watch `eval/rewards/accuracies`: the share of held-out pairs where the model now prefers the chosen briefing. It should climb from about 0.5 toward 0.8 or more.

In [ ]:
import os, time
from trl import DPOConfig, DPOTrainer

OUTPUT_DIR = os.path.join(WORK_DIR, OUTPUT_NAME + "-checkpoints")
cfg = DPOConfig(
    output_dir=OUTPUT_DIR,
    beta=0.1,
    num_train_epochs=2,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=1,
    eval_strategy="steps",
    eval_steps=10,
    logging_steps=5,
    save_strategy="steps",
    save_steps=20,
    save_total_limit=2,
    max_length=8192,
    max_prompt_length=7000,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    seed=3407,
    report_to="none",
)
trainer = DPOTrainer(model=model, ref_model=None, args=cfg, train_dataset=train_ds, eval_dataset=eval_ds, processing_class=tokenizer)
t0 = time.time()
trainer.train()
print(f"trained in {(time.time() - t0) / 60:.0f} minutes")
json.dump(trainer.state.log_history, open(os.path.join(WORK_DIR, OUTPUT_NAME + "-train-log.json"), "w"))
print("log saved")


## 5. Save the stage 2 adapter

Saved next to the stage 1 one. `02_generate_eval_answers.ipynb` can then score it: set `ADAPTER_DIR` there to `news-briefer-dpo-lora` and the answers file name to `dpo_answers.jsonl`.

In [ ]:
DPO_ADAPTER_DIR = os.path.join(WORK_DIR, OUTPUT_NAME)
model.save_pretrained(DPO_ADAPTER_DIR)
tokenizer.save_pretrained(DPO_ADAPTER_DIR)
print("saved", DPO_ADAPTER_DIR)

from google.colab import files
files.download(os.path.join(WORK_DIR, OUTPUT_NAME + "-train-log.json"))
